In [1]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv")
products = pd.read_csv("../data/products.csv")

In [2]:
import os

os.getcwd()

'c:\\Users\\RyanD\\OneDrive\\Documents\\GitHub\\cds492-instacart-project\\code'

In [3]:
os.listdir("../data")

['aisles.csv',
 'basket.csv',
 'departments.csv',
 'Groceries data.csv',
 'orders.csv',
 'order_products__prior.csv',
 'order_products__train.csv',
 'products.csv']

In [4]:
import pandas as pd
import numpy as np

In [5]:
# we start first with the instacart tables only
# then we read the data our data files and assign them to variables

orders = pd.read_csv("../data/orders.csv")
order_prior = pd.read_csv("../data/order_products__prior.csv")
products = pd.read_csv("../data/products.csv")
aisles = pd.read_csv("../data/aisles.csv")
departments = pd.read_csv("../data/departments.csv")

In [6]:
print("Orders shape:", orders.shape)
print("Order Prior shape:", order_prior.shape)
print("Products shape:", products.shape)

orders.head()

Orders shape: (3421083, 7)
Order Prior shape: (32434489, 4)
Products shape: (49688, 4)


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [7]:
# we check missing values for orders,oder_prior, and products datasets
# days_since_prior_order has a large values which means those are the customers who necer ordered before so it is 
# logicall to have this column filld with a null value "NaN"
orders.isnull().sum()

order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

In [8]:
order_prior.isnull().sum()

order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

In [9]:
products.isnull().sum()

product_id       0
product_name     0
aisle_id         0
department_id    0
dtype: int64

In [10]:
# removing duplicate values
orders = orders.drop_duplicates()
order_prior = order_prior.drop_duplicates()
products = products.drop_duplicates()

In [11]:
# standarize product names
#convert to lower case
#remove extra spaces at the begining and end of text
products["product_name"] = products["product_name"].str.lower().str.strip()

In [12]:
# we can check for unique values
# how many unique products are in the dataset
products["product_name"].nunique()

49588

In [13]:
# part 2
# we need to know what we are working with
# understand the datasets

print("Orders:", orders.shape)
print("Order-Product (Prior):", order_prior.shape)
print("Products:", products.shape)
print("Aisles:", aisles.shape)
print("Departments:", departments.shape)

Orders: (3421083, 7)
Order-Product (Prior): (32434489, 4)
Products: (49688, 4)
Aisles: (134, 2)
Departments: (21, 2)


In [14]:
orders["order_id"].nunique()

3421083

In [15]:
products["product_id"].nunique()

49688

In [16]:
# we can get a percentage of repeated orders this can help us with our business question too
# note: this column "reordered" has 1s and 0s. 1 means did order again 0 means did not
order_prior["reordered"].mean()

np.float64(0.5896974667922161)

In [17]:
order_prior_products = order_prior.merge(
    products,
    on="product_id",
    how="left"
)

order_prior_products.head()

,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id
0,2,33120,1,1,organic egg whites,86,16
1,2,28985,2,1,michigan organic kale,83,4
2,2,9327,3,0,garlic powder,104,13
3,2,45918,4,1,coconut butter,19,13
4,2,30035,5,0,natural sweetener,17,13


In [18]:
# quick schema check for all data files
import os

for filename in sorted(os.listdir("../data")):
    if filename.endswith(".csv"):
        df = pd.read_csv(f"../data/{filename}")
        print(f"\n{filename}")
        print("shape:", df.shape)
        print("columns:", list(df.columns))



Groceries data.csv
shape: (38765, 7)
columns: ['Member_number', 'Date', 'itemDescription', 'year', 'month', 'day', 'day_of_week']

aisles.csv
shape: (134, 2)
columns: ['aisle_id', 'aisle']

basket.csv
shape: (14963, 11)
columns: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10']

departments.csv
shape: (21, 2)
columns: ['department_id', 'department']

order_products__prior.csv
shape: (32434489, 4)
columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']

order_products__train.csv
shape: (1384617, 4)
columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']

orders.csv
shape: (3421083, 7)
columns: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']

products.csv
shape: (49688, 4)
columns: ['product_id', 'product_name', 'aisle_id', 'department_id']


## Unsupervised Modeling

This section builds, trains, tests, and evaluates unsupervised models using all CSV files in `../data`:
- Instacart user behavior segmentation
- Grocery member transaction segmentation
- Basket transaction segmentation

In [19]:
# Modeling and evaluation utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score


def _safe_cluster_metrics(X, labels):
    n_clusters = len(np.unique(labels))
    if n_clusters < 2:
        return {
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
            "n_clusters": n_clusters,
        }

    return {
        "silhouette": silhouette_score(X, labels),
        "calinski_harabasz": calinski_harabasz_score(X, labels),
        "davies_bouldin": davies_bouldin_score(X, labels),
        "n_clusters": n_clusters,
    }


def evaluate_unsupervised_dataset(feature_df, dataset_name, random_state=42):
    feature_df = feature_df.dropna().copy()

    X_train, X_test = train_test_split(feature_df, test_size=0.2, random_state=random_state)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Select k using train silhouette for KMeans
    k_grid = list(range(2, 11))
    k_search = []
    for k in k_grid:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        train_labels = km.fit_predict(X_train_scaled)
        k_search.append({
            "k": k,
            "train_silhouette": silhouette_score(X_train_scaled, train_labels),
            "train_inertia": km.inertia_,
        })

    k_search_df = pd.DataFrame(k_search).sort_values("train_silhouette", ascending=False)
    best_k = int(k_search_df.iloc[0]["k"])

    # Model 1: KMeans
    kmeans = KMeans(n_clusters=best_k, random_state=random_state, n_init=10)
    km_train_labels = kmeans.fit_predict(X_train_scaled)
    km_test_labels = kmeans.predict(X_test_scaled)

    km_train_metrics = _safe_cluster_metrics(X_train_scaled, km_train_labels)
    km_test_metrics = _safe_cluster_metrics(X_test_scaled, km_test_labels)

    # Model 2: Gaussian Mixture
    gmm = GaussianMixture(n_components=best_k, covariance_type="full", random_state=random_state)
    gmm.fit(X_train_scaled)
    gm_train_labels = gmm.predict(X_train_scaled)
    gm_test_labels = gmm.predict(X_test_scaled)

    gm_train_metrics = _safe_cluster_metrics(X_train_scaled, gm_train_labels)
    gm_test_metrics = _safe_cluster_metrics(X_test_scaled, gm_test_labels)

    rows = [
        {
            "dataset": dataset_name,
            "model": "KMeans",
            "best_k": best_k,
            "split": "train",
            **km_train_metrics,
        },
        {
            "dataset": dataset_name,
            "model": "KMeans",
            "best_k": best_k,
            "split": "test",
            **km_test_metrics,
        },
        {
            "dataset": dataset_name,
            "model": "GaussianMixture",
            "best_k": best_k,
            "split": "train",
            **gm_train_metrics,
            "gmm_aic": gmm.aic(X_train_scaled),
            "gmm_bic": gmm.bic(X_train_scaled),
        },
        {
            "dataset": dataset_name,
            "model": "GaussianMixture",
            "best_k": best_k,
            "split": "test",
            **gm_test_metrics,
            "gmm_aic": gmm.aic(X_test_scaled),
            "gmm_bic": gmm.bic(X_test_scaled),
        },
    ]

    result_df = pd.DataFrame(rows)

    return {
        "results": result_df,
        "k_search": k_search_df,
        "kmeans_model": kmeans,
        "gmm_model": gmm,
        "scaler": scaler,
        "X_train": X_train,
        "X_test": X_test,
        "X_train_scaled": X_train_scaled,
        "X_test_scaled": X_test_scaled,
        "kmeans_train_labels": km_train_labels,
        "kmeans_test_labels": km_test_labels,
    }


In [20]:
# Dataset 1: Instacart user-level features (uses orders, prior, train, products, aisles, departments)
order_train = pd.read_csv("../data/order_products__train.csv")

product_lookup = products[["product_id", "aisle_id", "department_id"]].copy()

all_order_products = pd.concat([
    order_prior[["order_id", "product_id", "add_to_cart_order", "reordered"]],
    order_train[["order_id", "product_id", "add_to_cart_order", "reordered"]],
], ignore_index=True)

op_with_user = all_order_products.merge(
    orders[["order_id", "user_id"]],
    on="order_id",
    how="left"
)

op_with_user = op_with_user.merge(product_lookup, on="product_id", how="left")

user_item_features = op_with_user.groupby("user_id").agg(
    total_items=("product_id", "count"),
    unique_products=("product_id", "nunique"),
    reorder_rate=("reordered", "mean"),
    avg_add_to_cart_order=("add_to_cart_order", "mean"),
    unique_aisles=("aisle_id", "nunique"),
    unique_departments=("department_id", "nunique"),
)

user_order_features = orders.groupby("user_id").agg(
    total_orders=("order_number", "max"),
    avg_order_dow=("order_dow", "mean"),
    avg_order_hour=("order_hour_of_day", "mean"),
    avg_days_since_prior=("days_since_prior_order", "mean"),
    std_days_since_prior=("days_since_prior_order", "std"),
)

instacart_user_features = user_order_features.join(user_item_features, how="inner")
instacart_user_features = instacart_user_features.fillna(0)

print("Instacart user feature shape:", instacart_user_features.shape)
instacart_user_features.head()


Instacart user feature shape: (206209, 11)


,total_orders,avg_order_dow,avg_order_hour,avg_days_since_prior,std_days_since_prior,total_items,unique_products,reorder_rate,avg_add_to_cart_order,unique_aisles,unique_departments
user_id,,,,,,,,,,,
1,11,2.636364,10.090909,19.000000,9.030811,70,19,0.728571,4.000000,13,7
2,15,2.066667,10.600000,16.285714,10.268912,226,121,0.464602,9.575221,37,13
3,13,1.384615,16.307692,12.000000,5.134553,88,33,0.625000,4.443182,16,9
4,6,4.500000,12.500000,17.000000,10.977249,18,17,0.055556,2.777778,14,9
5,5,1.400000,15.000000,11.500000,5.446712,46,28,0.391304,5.413043,17,10


In [21]:
# Dataset 2: Grocery member-level features (uses Groceries data.csv)
groceries = pd.read_csv("../data/Groceries data.csv")
groceries["Date"] = pd.to_datetime(groceries["Date"], errors="coerce")

grocery_member_features = groceries.groupby("Member_number").agg(
    total_transactions=("itemDescription", "count"),
    unique_items=("itemDescription", "nunique"),
    active_days=("Date", "nunique"),
    active_months=("month", "nunique"),
    avg_year=("year", "mean"),
    weekend_ratio=("day_of_week", lambda s: s.astype(str).str.lower().isin(["saturday", "sunday", "sat", "sun"]).mean()),
)

grocery_member_features["items_per_active_day"] = (
    grocery_member_features["total_transactions"] / grocery_member_features["active_days"].replace(0, np.nan)
)
grocery_member_features = grocery_member_features.fillna(0)

print("Grocery member feature shape:", grocery_member_features.shape)
grocery_member_features.head()


Grocery member feature shape: (3898, 7)


,total_transactions,unique_items,active_days,active_months,avg_year,weekend_ratio,items_per_active_day
Member_number,,,,,,,
1000,13,11,5,5,2014.769231,0.0,2.600
1001,12,9,5,5,2014.583333,0.0,2.400
1002,8,8,4,3,2014.500000,0.0,2.000
1003,8,6,4,3,2014.250000,0.0,2.000
1004,21,16,8,5,2014.095238,0.0,2.625


In [22]:
# Dataset 3: Basket transaction-level features (uses basket.csv)
basket = pd.read_csv("../data/basket.csv")

basket_long = (
    basket.reset_index(names="basket_id")
    .melt(id_vars="basket_id", value_name="item")
    .dropna(subset=["item"])
)

basket_long["item"] = basket_long["item"].astype(str).str.strip()
basket_long = basket_long[basket_long["item"] != ""]

basket_features = basket_long.groupby("basket_id").agg(
    basket_size=("item", "count"),
    unique_items=("item", "nunique"),
    avg_item_name_length=("item", lambda s: s.str.len().mean()),
)

basket_features["item_uniqueness_ratio"] = (
    basket_features["unique_items"] / basket_features["basket_size"].replace(0, np.nan)
)
basket_features = basket_features.fillna(0)

print("Basket feature shape:", basket_features.shape)
basket_features.head()


Basket feature shape: (14963, 4)


,basket_size,unique_items,avg_item_name_length,item_uniqueness_ratio
basket_id,,,,
0,3,3,9.0,1.0
1,4,4,10.5,1.0
2,2,2,11.0,1.0
3,2,2,13.0,1.0
4,2,2,11.5,1.0


In [ ]:
# Train + test models for each feature set
instacart_eval = evaluate_unsupervised_dataset(instacart_user_features, "Instacart Users")
grocery_eval = evaluate_unsupervised_dataset(grocery_member_features, "Grocery Members")
basket_eval = evaluate_unsupervised_dataset(basket_features, "Basket Transactions")

all_results = pd.concat([
    instacart_eval["results"],
    grocery_eval["results"],
    basket_eval["results"],
], ignore_index=True)

all_results.sort_values(["dataset", "model", "split"]).reset_index(drop=True)


In [ ]:
# Prediction metrics summary (test split only)
test_metrics = all_results[all_results["split"] == "test"].copy()
metric_cols = ["silhouette", "calinski_harabasz", "davies_bouldin"]

summary_table = test_metrics[["dataset", "model", "best_k"] + metric_cols].sort_values(
    ["dataset", "silhouette"], ascending=[True, False]
)

summary_table


In [ ]:
# Results: identify best model per dataset by silhouette (higher is better)
best_models = (
    summary_table.sort_values(["dataset", "silhouette"], ascending=[True, False])
    .groupby("dataset", as_index=False)
    .first()
)

best_models


In [ ]:
# Optional: quick cluster profile for Instacart users using best KMeans
instacart_test_profile = instacart_eval["X_test"].copy()
instacart_test_profile["cluster"] = instacart_eval["kmeans_test_labels"]

instacart_cluster_profile = instacart_test_profile.groupby("cluster").mean().round(2)
instacart_cluster_profile
